# Dataset 5: Thursday-Afternoon-Infiltration EDA
This notebook analyzes multi-stage Infiltration attacks (36 flows out of 288,602 records).


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../dataset/Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv", encoding='latin1', low_memory=False)
df.columns = [c.strip() for c in df.columns]

cols = list(df.columns)
if cols.count('Fwd Header Length') > 1:
    first_idx = cols.index('Fwd Header Length')
    second_idx = cols.index('Fwd Header Length', first_idx + 1)
    cols[second_idx] = 'Fwd Header Length.1'
    df.columns = cols

print(f"Loaded Thursday Infiltration: {df.shape[0]:,} rows")
print(f"Class breakdown:\n{df['Label'].value_counts()}")


## 1. Infiltration Behavioral Dynamics & Flag Activations

In [ ]:
for c in [c for c in df.columns if c != 'Label']:
    df[c] = pd.to_numeric(df[c], errors='coerce')

infil_df = df[df['Label'] == 'Infiltration']
print(f"Infiltration Destination Ports: {infil_df['Destination Port'].value_counts().to_dict()}")
print(f"Infiltration Median Flow Duration: {infil_df['Flow Duration'].median()/1000:.2f} ms")
print(f"Fwd URG Flags non-zero count: {(df['Fwd URG Flags'] > 0).sum()} (all in Benign)")

plt.figure(figsize=(8, 4.5))
df['log_Flow_Duration'] = np.log10(df['Flow Duration'].clip(lower=1))
sns.boxplot(data=df, x='Label', y='log_Flow_Duration', palette=['#2b5c8f', '#e41a1c'])
plt.title("Thursday Afternoon: Flow Duration (Benign vs Infiltration)")
plt.show()


## 2. Key Findings & Preprocessing Recommendations
- Infiltration is an ultra-rare class (36 flows, 0.0125%).
- 100% of Infiltration flows target Port 444 with long connection durations (median 93.2s).
- Unsupervised anomaly detection (Isolation Forest) is the recommended evaluation paradigm.
